In [ ]:
!pip install python-dotenv huggingface_hub datasets underthesea langdetect requests[socks]

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="huggingface_hub.*")

In [ ]:
import glob
import os
import random
import re
import threading
import time
import warnings
from abc import ABC, abstractmethod
from concurrent.futures import ThreadPoolExecutor, as_completed

import pandas as pd
import requests
from bs4 import BeautifulSoup
from dotenv import find_dotenv, load_dotenv
from google.colab import drive
from huggingface_hub import login
from requests.adapters import HTTPAdapter
from requests.exceptions import (
    ChunkedEncodingError,
    ConnectionError,
    RequestException
)
from tqdm.auto import tqdm
from urllib3.util.retry import Retry

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
root_path = "/content/drive/MyDrive/BTL_AI_13_IT2302/AIWritingIndicator-13/SourceCode/"
%cd {root_path}
!pwd

/content/drive/.shortcut-targets-by-id/17JBaRqvfDFY0j8cN0vwu8kzqwtDISdDR/BTL_AI_13_IT2302/AIWritingIndicator-13/SourceCode
/content/drive/.shortcut-targets-by-id/17JBaRqvfDFY0j8cN0vwu8kzqwtDISdDR/BTL_AI_13_IT2302/AIWritingIndicator-13/SourceCode


In [ ]:
env_path = find_dotenv()

if env_path:
    load_dotenv(env_path)
else:
    print("Không tìm thấy file .env!")

In [ ]:
hf_token = os.getenv("HF_TOKEN")

if hf_token:
    login()
else:
    print("Không tồn tại env HF_TOKEN!")

### Giải thích Source Code

Đoạn mã bên dưới triển khai một hệ thống thu thập tin tức tự động từ các báo điện tử lớn (**VnExpress, Thanh Niên, VietnamPlus**) với các đặc điểm chính:

1.  **Mục tiêu:** Thu thập các bài báo tiếng Việt chất lượng (độ dài 200-800 từ) để làm dữ liệu huấn luyện/kiểm thử cho mô hình AI.
2.  **Cơ chế hoạt động:**
    *   Sử dụng chiến lược (Strategy Pattern) riêng biệt cho từng tờ báo để bóc tách nội dung chính xác.
    *   Tìm kiếm bài viết theo từ khóa và lọc theo thời gian (ưu tiên dữ liệu cũ trước 2022 để tránh nhiễu từ AI).
3.  **Kỹ thuật chống chặn (Anti-Ban):**
    *   Sử dụng `AntiBanRequestManager` để quản lý tần suất yêu cầu (Rate Limiting).
    *   Tự động thay đổi User-Agent và hỗ trợ Proxy để tránh bị khóa IP.
    *   Cơ chế thử lại (Retry) với độ trễ tăng dần (Exponential Backoff).
4.  **Lưu trữ:** Dữ liệu được xử lý làm sạch và lưu thành các file CSV theo từng lô (batch) 250 bài để tối ưu bộ nhớ.

In [ ]:
def count_words(text):
    return len(text.split())

class AntiBanRequestManager:
    def __init__(self, max_req_per_sec=5, max_retries=5, proxies_list=None):
        self.max_retries = max_retries
        self.proxies_list = proxies_list or []
        self.session = requests.Session()

        retry_strategy = Retry(
            total=3,
            backoff_factor=1,
            status_forcelist=[429, 500, 502, 503, 504],
            allowed_methods=["HEAD", "GET", "OPTIONS"]
        )

        adapter = HTTPAdapter(
            pool_connections=50,
            pool_maxsize=50,
            max_retries=retry_strategy
        )

        self.session.mount("https://", adapter)
        self.session.mount("http://", adapter)

        self.max_req_per_sec = max_req_per_sec
        self.min_delay = 1.0 / max_req_per_sec if max_req_per_sec > 0 else 0
        self.last_request_time = 0
        self.rate_lock = threading.Lock()

        self.user_agents = [
            'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
            'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/17.2.1 Safari/605.1.15',
            'Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:121.0) Gecko/20100101 Firefox/121.0',
            'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/119.0.0.0 Safari/537.36',
            'Mozilla/5.0 (iPhone; CPU iPhone OS 17_2 like Mac OS X) AppleWebKit/605.1.15 (KHTML, like Gecko) CriOS/120.0.6099.119 Mobile/15E148 Safari/604.1'
        ]

    def _enforce_polite_rate(self):
        with self.rate_lock:
            now = time.time()
            elapsed = now - self.last_request_time
            if elapsed < self.min_delay:
                time.sleep(self.min_delay - elapsed)
            time.sleep(random.uniform(0.05, 0.2))
            self.last_request_time = time.time()

    def _get_headers(self):
        return {
            'User-Agent': random.choice(self.user_agents),
            'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8',
            'Accept-Language': 'vi-VN,vi;q=0.9,en-US;q=0.8,en;q=0.7',
            'Connection': 'keep-alive',
            'Upgrade-Insecure-Requests': '1',
            'Sec-Fetch-Dest': 'document',
            'Sec-Fetch-Mode': 'navigate',
            'Sec-Fetch-Site': 'none'
        }

    def fetch(self, url):
        for attempt in range(1, self.max_retries + 1):
            self._enforce_polite_rate()
            headers = self._get_headers()
            proxy = {"http": random.choice(self.proxies_list), "https": random.choice(self.proxies_list)} if self.proxies_list else None

            try:
                res = self.session.get(url, headers=headers, proxies=proxy, timeout=15)

                if res.status_code in [403, 429]:
                    backoff_time = (2 ** attempt) + random.uniform(0, 1)
                    time.sleep(backoff_time)
                    continue

                if res.status_code == 200:
                    if not res.text or len(res.text.strip()) < 500:
                        time.sleep(2)
                        continue
                    return res

                time.sleep(2)

            except RequestException:
                time.sleep(2)

        return None

In [ ]:
class NewsStrategy(ABC):
    @property
    @abstractmethod
    def source_name(self): pass

    @abstractmethod
    def get_url(self, page): pass

    @abstractmethod
    def extract_links(self, soup): pass

    @abstractmethod
    def extract_content(self, soup): pass

class VnExpressStrategy(NewsStrategy):
    @property
    def source_name(self): return 'VnExpress'

    def get_url(self, page):
        keywords = ["tai nạn", "bắt giữ", "điều tra", "tòa án", "cảnh sát", "lãi suất", "ngân hàng", "bất động sản", "chứng khoán", "doanh nghiệp", "AI", "trí tuệ nhân tạo", "startup", "công nghệ", "ứng dụng", "giá vàng", "thời tiết", "du lịch", "ẩm thực", "giá xăng", "Mỹ", "Trung Quốc", "Nga", "chiến tranh", "ngoại giao", "đại học", "thi tốt nghiệp", "học sinh", "giáo viên"]
        current_keyword = keywords[((page - 1) // 50) % len(keywords)]
        actual_page = ((page - 1) % 50) + 1
        return f"https://timkiem.vnexpress.net/?q={current_keyword}&fromdate=0&todate=1640995200&page={actual_page}"

    def extract_links(self, soup):
        return [a['href'] for a in soup.select('h3.title-news a')]

    def extract_content(self, soup):
        article = soup.select_one('article.fck_detail') or soup.select_one('article.content-detail')
        if article:
            for tag in article.select('table, figure, picture, .box-item-video, .video-player, ul.list-news, p.author_mail, p[style*="text-align:right"], p[align="right"]'):
                tag.decompose()
            return article.get_text(separator=' ', strip=True)
        return ""

class ThanhNienStrategy(NewsStrategy):
    @property
    def source_name(self): return 'ThanhNien'

    def extract_pub_date(self, soup):
        meta_date = soup.find('meta', property='article:published_time')
        if meta_date: return meta_date.get('content')

        time_tag = soup.select_one('.detail-time, time')
        return time_tag.get_text(strip=True) if time_tag else None

    def get_url(self, page):
        keywords = ["tai nạn", "bắt giữ", "điều tra"]
        current_keyword = keywords[((page - 1) // 5) % len(keywords)]
        actual_page = page
        return f"https://thanhnien.vn/tim-kiem.htm?keywords={current_keyword}&page={actual_page}"

    def extract_links(self, soup):
        return list(set([
            "https://thanhnien.vn" + a['href'] if not a['href'].startswith('http') else a['href']
            for a in soup.select('a[href*=".htm"]')
            if len(a['href']) > 35 and 'trang-' not in a['href']
        ]))

    def extract_content(self, soup):
        pub_date = self.extract_pub_date(soup)
        if pub_date and any(yr in pub_date for yr in ["2023", "2024", "2025", "2026"]):
            return ""

        article = soup.select_one('.detail-content, #abody, .cms-body')
        if article:
            return article.get_text(separator=' ', strip=True)
        return ""

In [ ]:
class VietnamPlusStrategy(NewsStrategy):
    @property
    def source_name(self): return 'VietnamPlus'

    def get_url(self, page):
        keywords = ["tai-nan", "bat-giu", "dieu-tra", "toa-an", "canh-sat", "lai-suat", "ngan-hang", "bat-dong-san", "chung-khoan", "doanh-nghiep", "AI", "tri-tue-nhan-tao", "startup", "cong-nghe", "ung-dung", "gia-vang", "thoi-tiet", "du-lich", "am-thuc", "gia-xang", "My", "Trung-Quoc", "Nga", "chien-tranh", "ngoai-giao", "dai-hoc", "thi-tot-nghiep", "hoc-sinh", "giao-vien"]
        current_keyword = keywords[((page - 1) // 50) % len(keywords)]
        actual_page = ((page - 1) % 50) + 1
        return f"https://www.vietnamplus.vn/tags/{current_keyword}/trang-{actual_page}.vnp"

    def extract_links(self, soup):
        links = []
        selectors = [
            'article.story a[href]',
            '.story__title a[href]',
            'h2.story__heading a[href]',
            'h3.story__heading a[href]'
        ]
        for sel in selectors:
            for a in soup.select(sel):
                href = a.get('href')
                if href and '.vnp' in href:
                    if not href.startswith('http'):
                        href = 'https://www.vietnamplus.vn' + href
                    links.append(href)
        return list(set(links))

    def extract_content(self, soup):
        pub_date_meta = soup.find('meta', property='article:published_time')
        if pub_date_meta and pub_date_meta.get('content'):
            try:
                pub_year = int(pub_date_meta['content'][:4])
                if pub_year >= 2023:
                    return ""
            except: pass

        article = soup.select_one('div.article-body, div.Details__Content, .article-content, #mainContent')
        sapo = soup.select_one('div.article-summary, div.Details__Summary, .sapo')

        if article:
            for tag in article.select('figure, video, .box-related, script, style, table, .VCSortableInPreviewMode'):
                tag.decompose()

            sapo_text = sapo.get_text(strip=True) if sapo else ""
            main_content = article.get_text(separator=' ', strip=True)
            return f"{sapo_text} {main_content}".strip()
        return ""

In [ ]:
class VietnameseNewsCrawler:
    def __init__(self, output_dir='Data/data_crawl/news', batch_size=250, max_workers=30, target_batches=20, max_req_per_sec=20, proxies=None, sources=None):
        self.output_dir = output_dir
        self.batch_size = batch_size
        self.max_workers = max_workers
        self.target_batches = target_batches
        self.proxies_list = proxies if proxies else []
        self.sources = sources if sources else ['VnExpress', 'ThanhNien', 'VietnamPlus']

        self.request_manager = AntiBanRequestManager(
            max_req_per_sec=max_req_per_sec,
            max_retries=5,
            proxies_list=self.proxies_list
        )

        self.buffers = {src: [] for src in self.sources}
        self.source_file_counts = {src: 0 for src in self.sources}
        self.crawled_urls = set()

        print("=" * 60)
        self._load_existing_data()
        print(f"Đã nạp {len(self.crawled_urls)} URL từ DB cũ để xét trùng lặp.")
        if not self.proxies_list: print("Đang dùng IP THẬT.")
        else: print(f"Đã nạp {len(self.proxies_list)} proxy.")
        print("=" * 60)

        strategy_dict = {'VnExpress': VnExpressStrategy(), 'ThanhNien': ThanhNienStrategy(), 'VietnamPlus': VietnamPlusStrategy()}
        self.active_strategies = [strategy_dict[src] for src in self.sources if src in strategy_dict]

    def _load_existing_data(self):
        if not os.path.exists(self.output_dir):
            return
        for root, _, files in os.walk(self.output_dir):
            for file in files:
                if file.endswith('.csv'):
                    try:
                        df = pd.read_csv(os.path.join(root, file))
                        if 'url' in df.columns:
                            self.crawled_urls.update(df['url'].dropna().tolist())
                    except Exception:
                        pass

    def _clean_text(self, text):
        if not text: return ""
        text = text.replace('\xa0', ' ').replace('\u200b', '')
        text = re.sub(r'(?:^|\.\s+)(?:>>\s*)?(?:xem thêm|đọc thêm|tin liên quan|mời xem tiếp|bài liên quan)\b.*?(?:\.|$)', '.', text, flags=re.IGNORECASE)
        text = re.sub(r'\(Theo\s.*?\)$|Nguồn:.*$|Ảnh:.*$|Bài, ảnh:.*$|Đồ họa:.*$', '', text, flags=re.IGNORECASE | re.MULTILINE)
        text = re.sub(r'^[A-Z\s]{2,15}\s?-\s?', '', text)
        text = re.sub(r'\[(VIDEO|CLIP|ẢNH)\][^A-ZÀ-Ỹ]*', '', text)
        text = re.sub(r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,7}\b', '', text)
        text = re.sub(r'(\+84|0)\s?\d{2,4}[-\s]?\d{3,4}[-\s]?\d{3,4}', '', text)
        text = re.sub(r'\b(?:https?://|www\.)[a-zA-Z0-9-]+\.[a-zA-Z0-9.-/]+\b', '', text)
        text = re.sub(r'\s+', ' ', text)
        return text.strip()

    def _is_valid(self, text):
        if not isinstance(text, str) or not text.strip(): return False
        words = text.split()
        if not (200 <= len(words) <= 800): return False
        if not re.search(r'[.!?]', text): return False
        return True

    def _get_filename(self, source_name):
        brand_path = os.path.join(self.output_dir, source_name)
        os.makedirs(brand_path, exist_ok=True)
        existing_files = glob.glob(os.path.join(brand_path, f"vietnamese_{source_name.lower()}_*.csv"))
        max_idx = max([int(re.search(r'_(\d+)\.csv$', f).group(1)) for f in existing_files if re.search(r'_(\d+)\.csv$', f)] + [0])
        return os.path.join(brand_path, f"vietnamese_{source_name.lower()}_{max_idx + 1}.csv")

    def _make_request(self, url):
        return self.request_manager.fetch(url)

    def _fetch_article(self, url, strategy):
        try:
            res = self._make_request(url)

            if res and res.status_code == 200:
                soup = BeautifulSoup(res.text, 'html.parser')
                try:
                    content = strategy.extract_content(soup)
                except Exception as parse_err:
                    print(f"[Lỗi Parse HTML] {url} | {parse_err}")
                    return None

                if content:
                    cleaned = self._clean_text(content)
                    if self._is_valid(cleaned):
                        return {'source': strategy.source_name, 'content': cleaned, 'url': url}
            return None
        except Exception as e:
            print(f"[Lỗi Trích xuất] {url} | {e}")
            return None

    def _save_batch(self, source_name):
        if len(self.buffers[source_name]) >= self.batch_size:
            filename = self._get_filename(source_name)
            df = pd.DataFrame(self.buffers[source_name][:self.batch_size])

            try:
                df.to_csv(filename, index=False, encoding='utf-8-sig')
                self.crawled_urls.update(df['url'].tolist())
                self.buffers[source_name] = self.buffers[source_name][self.batch_size:]
                self.source_file_counts[source_name] += 1
            except Exception as e:
                print(f"\n[LỖI NGHIÊM TRỌNG] Không thể lưu file {filename} | Chi tiết: {e}")

    def _get_links_from_search(self, strategy, page):
        res = self._make_request(strategy.get_url(page))
        if res and res.status_code == 200:
            soup = BeautifulSoup(res.text, 'html.parser')
            return strategy, strategy.extract_links(soup)
        return strategy, []

    def run(self):
        print("Đang tối ưu bộ nhớ và nạp URL cũ...")
        page_trackers = {s: 1 for s in self.active_strategies}
        progress_bars = {s.source_name: tqdm(total=self.target_batches * self.batch_size,
                             desc=f"{s.source_name:12}", unit=" bài") for s in self.active_strategies}

        with ThreadPoolExecutor(max_workers=self.max_workers) as executor:
            while self.active_strategies:
                search_tasks = []

                for strategy in list(self.active_strategies):
                    src = strategy.source_name
                    if self.source_file_counts[src] >= self.target_batches:
                        self.active_strategies.remove(strategy)
                        continue

                    search_tasks.append(executor.submit(self._get_links_from_search, strategy, page_trackers[strategy]))
                    page_trackers[strategy] += 1

                links_to_fetch = []
                for future in as_completed(search_tasks):
                    strat, links = future.result()
                    if links:
                        for l in links:
                            if l not in self.crawled_urls:
                                links_to_fetch.append((strat, l))
                                self.crawled_urls.add(l)

                if not links_to_fetch:
                    print("Không tìm thấy link mới, đang xoay vòng keyword...")
                    time.sleep(2)
                    continue

                content_tasks = {executor.submit(self._fetch_article, url, strat): (strat.source_name, url)
                                 for strat, url in links_to_fetch}

                for future in as_completed(content_tasks):
                    src_name, url = content_tasks[future]
                    try:
                        result_data = future.result()

                        if result_data and isinstance(result_data, dict):
                            self.buffers[src_name].append(result_data)
                            progress_bars[src_name].update(1)
                            self._save_batch(src_name)

                    except Exception as e:
                        tqdm.write(f"[Lỗi Worker] {src_name} - {url} | Chi tiết: {str(e)}")

In [ ]:
crawler = VietnameseNewsCrawler(
  output_dir='Data/data_crawl/news',
  batch_size=250,
  max_workers=20,
  target_batches=20,
  max_req_per_sec=20,
  sources=['VnExpress', 'ThanhNien', 'VietnamPlus']
)
crawler.run()